## 강화학습 3장. Dynamic Programming

* 출처 1. https://with-rl.tistory.com/entry/%EB%B0%94%EB%8B%A5%EB%B6%80%ED%84%B0-%EB%B0%B0%EC%9A%B0%EB%8A%94-%EA%B0%95%ED%99%94-%ED%95%99%EC%8A%B5-04-MDP%EB%A5%BC-%EC%95%8C-%EB%95%8C%EC%9D%98-%ED%94%8C%EB%9E%98%EB%8B%9D
* 출처 2. https://github.com/seungeunrho/RLfrombasics/tree/master

In [52]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from typing import *

---

### 3.1. 본 수업 : OpenAI Gym의 FrozenLake-v1 환경에서 정책 평가와 정책 반복 알고리즘을 구현

In [6]:
# Temporal Difference Learning [TD]

"""
구현을 위해 필요한 요소들
1. Grid World 환경 직접 구현 → gymnasium FrozenLake-v1 환경 활용(바닥이 미끄러운 얼음 호수 건너기)
2. TD / MC 밸류 테이블 채우기 → 4 by 4 격자
3. 랜덤 정책(밸류에서 정책 뽑기, 어느 방향이 제일 높은지 argmax) → 정책 평가 
4. 정책 반복(평가, 개선, 평가, 개선, ...)
5. TD / MC  → MDP 모델이 없는 경우(모르는 경우) 학습
"""

# Monte Carlo [MC]

'\n구현을 위해 필요한 요소들\n1. Grid World 환경 직접 구현 → gymnasium FrozenLake-v1 환경 활용(바닥이 미끄러운 얼음 호수 건너기)\n2. TD / MC 밸류 테이블 채우기 → 4 by 4 격자\n3. 랜덤 정책(밸류에서 정책 뽑기, 어느 방향이 제일 높은지 argmax) → 정책 평가 \n4. 정책 반복(평가, 개선, 평가, 개선, ...)\n5. TD / MC  → MDP 모델이 없는 경우(모르는 경우) 학습\n'

---

### 3.2. 벨만 방정식과 정책 평가 알고리즘 [결정론적]
* 반복적 정책 평가(iterative policy evaluation) 방법을 통해 각 상태 $s$에 대한 가치 함수 $v(s)$ 계산 가능

In [23]:
gamma: float = 0.9
# 할인율

def policyEvaluation(env, policy):
    """랜덤 정책을 벨만 방정식을 이용하여 평가하는 메서드
    args :
    policy 정책 pi

    environment : 
    Frozen(F) 호수 위를 걸어가면서 Holes(H)에 빠지지 않고 
    Start(S)에서 골(G)까지 얼어붙은 호수를 건너는 것을 목표

    logics:
    정책 반복 알고리즘

    returns :
    V 가치함수 v_n array
    """
    V = np.zeros( env.observation_space.n ) 
    # 가치함수를 저장할 표 생성

    while True:
        oldV = V.copy()
        # 현재 가치함수를 oldV에 복사하여 저장(이전 가치함수)

        for state in range( env.observation_space.n ):
            # 각 상태에 대해
            v = 0
            # 현재 상태에서의 가치 초기화

            for action, action_prob in enumerate(policy[state]):
                # 벨만 방정식(순환식)
                prob, next_state, reward, terminated = env.unwrapped.P[state][action][0]
                # 좌변: 새로운 상태(state), 보상(reward), 종료 여부(terminated), 타임아웃 여부(truncated)
                # 우변: 마르코프 의사결정 프로세스상 전이 확률

                # [FIX] action_prob(...)  →  action_prob * (...)
                # action_prob는 float(확률값)이므로 곱셈(*)으로 연산해야 함
                # 원본 코드의 action_prob(...)는 함수 호출 문법으로, TypeError 발생
                v = v + action_prob * (reward + gamma * V[next_state])
                # [핵심] 현재 상태의 가치 = 현재 상태에서 특정 행동을 할 확률(왼오위아래) x (보상 + 감쇠인자 * 다음 상태의 가치)
            
            V[state] = v
            # 현재 상태의 가치(값) 표에 저장

        if max( np.abs(V - oldV) ) < 1e-8:
            # [핵심] 변화가 충분히 작으면 수렴으로 간주(self-consistency, bootstrapping)
            break

    # [FIX] return V 추가
    # 원본 코드에는 return이 없어서 None을 반환함
    # 호출부에서 V_01 = policyEvaluation(...)으로 받으므로 반드시 반환해야 함
    return V

### 3.3.1. 벨만 최적 방정식과 정책 반복(개선) 알고리즘 [결정론적]

In [25]:
gamma: float = 0.9
# 할인율

def policyIteration(env):
    """랜덤 정책을 벨만 방정식을 이용하여 평가하고 개선 단계를 반복하여 최적 정책으로 접근하는 메서드
    args:
    env 과업

    environment : 
    Frozen(F) 호수 위를 걸어가면서 Holes(H)에 빠지지 않고 
    Start(S)에서 골(G)까지 얼어붙은 호수를 건너는 것을 목표

    logics:
    정책 반복 알고리즘

    returns :
    - V 최적가치함수 v_n
    - pi 최적정책 pi*
    """
    V = np.zeros( env.observation_space.n )
    # 가치함수를 저장할 표 생성
    pi = [0 for i in range( env.observation_space.n )]
    # 정책을 행동 0으로 초기화

    while True:
        # 1+2. 정책 반복(Generalized Policy Iteration strategy)
        while True:
            # 1. 정책 평가 단계(Evaluation)
            oldV = V.copy()
            # 현재 가치함수를 oldV에 복사하여 저장(이전 가치함수)

            for state in range( env.observation_space.n ):
                # 각 상태에 대해
                action = pi[state]
                # [핵심] 현재 상태에서의 최적의 행동을 할 확률(최적 정책)
                # [as-is] for action, action_prob in enumerate(policy[state]): # 벨만 방정식(순환식)
                
                prob, next_state, reward, terminated = env.unwrapped.P[state][action][0]
                # 좌변: 새로운 상태(state), 보상(reward), 종료 여부(terminated), 타임아웃 여부(truncated)
                # 우변: 마르코프 의사결정 프로세스상 전이 확률
                v = reward + gamma * V[next_state]
                # [핵심] 현재 상태의 가치 = (보상 + 감쇠인자 * 다음 상태의 가치)
                
                V[state] = v
                # 현재 상태의 가치(값) 표에 저장

            if max( np.abs(V - oldV) ) < 1e-8:
                # [핵심] 변화가 충분히 작으면 수렴으로 간주(self-consistency, bootstrapping)
                break
        
        converged = True
        # 2. 정책 개선 단계(Iteration)
        for state in range( env.observation_space.n ):
            # 각 상태에 대해
            old_action = pi[state]
            # 
            q = np.zeros( env.action_space.n )
            # 탐욕적인 선택(greedy), 정책을 2차원이 아닌 1차원 표에 저장

            for action in range( env.action_space.n ):
            # 각 행동 공간에서의 행동에 대해
                prob, next_state, reward, terminated = env.unwrapped.P[state][action][0]
                # 좌변: 새로운 상태(state), 보상(reward), 종료 여부(terminated), 타임아웃 여부(truncated)
                # 우변: 마르코프 의사결정 프로세스상 전이 확률
                q[action] = reward + gamma * V[next_state]
                # [핵심] 기댓값 연산 = (보상 + 감쇠인자 * 다음 상태의 가치)

            # [FIX] pi[state] = np.argmax(q)를 for action 루프 밖으로 이동
            # 원본은 루프 안에서 action 하나씩 볼 때마다 정책을 갱신했으나,
            # 모든 action의 q값을 다 채운 뒤 argmax를 해야 올바른 최적 행동을 고를 수 있음
            pi[state] = np.argmax(q)
            # 개선된 정책으로 업데이트

            if pi[state] != old_action:
                converged = False
                # 변화가 없으면 수렴으로 간주하고 루프 중단

        # [FIX] if converged / return을 for state 루프 밖(while True 루프 안)으로 이동
        # 원본은 for action 루프 안에서 break/return이 실행되어,
        # 첫 번째 state의 첫 번째 action만 보고 함수가 종료되는 심각한 버그가 있었음
        # 모든 state를 다 순회한 뒤 수렴 여부를 판단해야 함
        if converged:
            break

    return pi, V
    # 최적 정책과 최적 가치 함수 반환


### 3.3.2. 벨만 최적 방정식과 가치 반복 알고리즘 [결정론적]

In [35]:
gamma: float = 0.9
# 할인율

def valueIteration(env):
    """정책 없이 벨만 방정식을 이용하여 가치 개선 단계를 반복하여 최적 정책으로 접근하는 메서드
    args:
    env 과업

    environment : 
    Frozen(F) 호수 위를 걸어가면서 Holes(H)에 빠지지 않고 
    Start(S)에서 골(G)까지 얼어붙은 호수를 건너는 것을 목표

    logics:
    가치 반복 알고리즘

    returns :
    - V 최적가치함수 v_n
    - pi 최적정책 pi*
    """
    V = np.zeros(env.observation_space.n) 
    # 가치함수를 저장할 표 생성
    
    while True:
    # 1. 가치 개선 단계
        oldV = V.copy()
        # 현재 가치함수를 oldV에 복사하여 저장(이전 가치함수)
        
        for state in range( env.observation_space.n ):
        # 각 상태에 대해
            q = np.zeros( env.action_space.n )
            # 탐욕적인 선택(greedy), 정책을 2차원이 아닌 1차원 표에 저장

            for action in range( env.action_space.n ):
            # 각 행동 공간에서의 행동에 대해
                prob, next_state, reward, terminated = env.unwrapped.P[state][action][0]
                # 좌변: 새로운 상태(state), 보상(reward), 종료 여부(terminated), 타임아웃 여부(truncated)
                # 우변: 마르코프 의사결정 프로세스상 전이 확률
                q[action] = reward + gamma * V[next_state]
                # [핵심] 기댓값 연산 = (보상 + 감쇠인자 * 다음 상태의 가치)

            # [FIX] V[state] = np.max(q)를 for action 루프 밖으로 이동
            # 원본은 action 루프 안에 있어서 q가 완성되기 전(action 하나씩 채울 때마다)
            # max를 구했으므로, 나머지 action들의 q값이 반영되지 않는 버그 발생
            # 모든 action에 대해 q를 다 채운 뒤 max를 구해야 올바른 벨만 최적 방정식
            V[state] = np.max(q)
            # [핵심] q의 최댓값을 표에 저장

        # [FIX] 수렴 체크를 for state 루프 밖으로 이동
        # 원본은 for state 루프 안에서 수렴 체크를 하여, 일부 state만 업데이트된 채로
        # while 루프가 종료될 수 있었음
        # 모든 state를 다 업데이트한 뒤 수렴 여부를 판단해야 함
        if max( np.abs(V - oldV) ) < 1e-8:
            # [핵심] 변화가 충분히 작으면 수렴으로 간주(self-consistency, bootstrapping)
            break
        
    pi = env.observation_space.n * [None]
    # 2. 최적 가치 함수로부터 최적 정책 구하는 단계
    for state in range( env.observation_space.n ):
        # 각 상태에 대해
        q = np.zeros( env.action_space.n )
        # 정책 저장할 표 생성

        for action in range( env.action_space.n ):
        # 각 행동 공간에서의 행동에 대해
            prob, next_state, reward, terminated = env.unwrapped.P[state][action][0]
            # 좌변: 새로운 상태(state), 보상(reward), 종료 여부(terminated), 타임아웃 여부(truncated)
            # 우변: 마르코프 의사결정 프로세스상 전이 확률
            q[action] = reward + gamma * V[next_state]
            # [핵심] 기댓값 연산 = (보상 + 감쇠인자 * 다음 상태의 가치)

        pi[state] = np.argmax(q)
        # [핵심] 최적 정책

    return pi, V
    # 최적 정책과 최적 가치 함수 반환

### 3.4. 스토캐스틱 MDP를 위한 가치 반복 알고리즘

In [39]:
gamma: float = 0.9
# 할인율

def stocasticValueIteration(env):
    """스토캐스틱 환경에서 정책 없이 벨만 방정식을 이용하여 가치 개선 단계를 반복하여 최적 정책으로 접근하는 메서드
    args:
    env 과업

    environment : 
    Frozen(F) 호수 위를 걸어가면서 Holes(H)에 빠지지 않고 
    Start(S)에서 골(G)까지 얼어붙은 호수를 건너는 것을 목표

    logics:
    MDP를 모르는 환경에서 가치 반복 알고리즘

    returns :
    - V 최적가치함수 v_n
    - pi 최적정책 pi*
    """
    V = np.zeros(env.observation_space.n) 
    # 가치함수를 저장할 표 생성
    
    while True:
    # 1. 가치 개선 단계
        oldV = V.copy()
        # 현재 가치함수를 oldV에 복사하여 저장(이전 가치함수)
        
        for state in range( env.observation_space.n ):
        # 각 상태에 대해
            q = np.zeros( env.action_space.n )
            # 탐욕적인 선택(greedy), 정책을 2차원이 아닌 1차원 표에 저장

            for action in range( env.action_space.n ):
            # 각 행동 공간에서의 행동에 대해
                for prob, next_state, reward, terminated in env.unwrapped.P[state][action]:
                # [핵심] 새로운 상태(state), 보상(reward), 종료 여부(terminated), 타임아웃 여부(truncated)

                    # [FIX] prob(...)  →  prob * (...)
                    # prob은 float(전이 확률값)이므로 곱셈(*)으로 연산해야 함
                    # 스토캐스틱 환경에서는 같은 action이라도 여러 next_state로 이동할 수 있으므로
                    # 각 전이 결과에 전이 확률을 곱해 기댓값을 누산(+=)함
                    q[action] = q[action] + prob * (reward + gamma * V[next_state])
                    # [핵심] 기댓값 연산 = 확률 * (보상 + 감쇠인자 * 다음 상태의 가치)

            # [FIX] V[state] = np.max(q)를 for action 루프 밖으로 이동 (valueIteration과 동일한 이유)
            V[state] = np.max(q)
            # q의 최댓값을 표에 저장

        # [FIX] 수렴 체크를 for state 루프 밖으로 이동 (valueIteration과 동일한 이유)
        if max( np.abs(V - oldV) ) < 1e-8:
            # 변화가 충분히 작으면 수렴으로 간주(self-consistency, bootstrapping)
            break
        
    pi = env.observation_space.n * [None]
    # 2. 최적 가치 함수로부터 최적 정책 구하는 단계
    for state in range( env.observation_space.n ):
        # 각 상태에 대해
        q = np.zeros( env.action_space.n )
        # 정책 저장할 표 생성

        for action in range( env.action_space.n ):
        # 각 행동 공간에서의 행동에 대해
            for prob, next_state, reward, terminated in env.unwrapped.P[state][action]:
            # [핵심]
                q[action] = q[action] + prob * (reward + gamma * V[next_state])
                # [핵심] 기댓값 연산 = 확률 * (보상 + 감쇠인자 * 다음 상태의 가치)

        pi[state] = np.argmax(q)
        # [핵심] 최적 정책

    return pi, V
    # 최적 정책과 최적 가치 함수 반환

### 3.2~3.4. 결과 비교 [as-is]

In [40]:
env_deterministic = gym.make(
    "FrozenLake-v1",
    is_slippery = False,
    render_mode = "ansi"
) # determine

In [41]:
pi_01 = np.ones(
    (env_deterministic.observation_space.n, env_deterministic.action_space.n)) / env_deterministic.action_space.n
# 랜덤 정책

In [42]:
V_01 = policyEvaluation(env_deterministic, pi_01)

In [43]:
pi_02, V_02= policyIteration(env_deterministic)

In [44]:
pi_03, V_03= valueIteration(env_deterministic)

In [46]:
env_stocastic = gym.make(
    "FrozenLake-v1",
    is_slippery = True,
    render_mode = "ansi"
) # stocastic

In [47]:
pi_04, V_04 = stocasticValueIteration(env_stocastic)

In [53]:
for V_i in [V_01, V_02, V_03, V_04]:
    print(V_i.reshape(4,4))

[[0.00447724 0.00422245 0.01006675 0.00411821]
 [0.00672195 0.         0.02633371 0.        ]
 [0.01867615 0.05760701 0.10697195 0.        ]
 [0.         0.13038305 0.39149016 0.        ]]
[[0.59049 0.6561  0.729   0.6561 ]
 [0.6561  0.      0.81    0.     ]
 [0.729   0.81    0.9     0.     ]
 [0.      0.9     1.      0.     ]]
[[0.59049 0.6561  0.729   0.6561 ]
 [0.6561  0.      0.81    0.     ]
 [0.729   0.81    0.9     0.     ]
 [0.      0.9     1.      0.     ]]
[[0.06889086 0.06141454 0.07440974 0.0558073 ]
 [0.09185451 0.         0.1122082  0.        ]
 [0.14543633 0.24749694 0.29961758 0.        ]
 [0.         0.37993589 0.63902014 0.        ]]


---

### 4.1. 에피소드 데이터 수집

In [50]:
def generateEpisode(env, policy):
    """에피소드 데이터 발생"""
    episode = []
    state, info = env.reset()
    while True:
        # [FIX] [1,2,3,4] → env.action_space.n (range 0~3)
        # FrozenLake의 action 공간은 0(LEFT), 1(DOWN), 2(RIGHT), 3(UP)으로 0~3임
        # [1,2,3,4]는 범위를 벗어난 잘못된 action 번호이고,
        # env.action_space.n을 쓰면 환경이 바뀌어도 자동으로 맞는 범위를 사용할 수 있음
        action = np.random.choice(
            env.action_space.n,
            p = policy[state]
        )

        # [FIX] state_1 → next_state (변수명 의미 명확화)
        next_state, reward, terminated, truncated, info = env.step(action)
        episode.append([state, action, reward])

        # [FIX] s = state 제거, state = next_state 로 상태 업데이트
        # 원본 코드는 state를 업데이트하지 않아 에이전트가 항상 같은 state에 머물렀음
        # 또한 루프 종료 조건이 없어 무한루프가 발생했음
        state = next_state

        # [FIX] 에피소드 종료 조건 추가
        # terminated: 목표 도달 또는 구멍에 빠짐 / truncated: 최대 스텝 초과
        if terminated or truncated:
            break

    # [FIX] return episode 추가
    # 원본 코드에는 return이 없어서 None을 반환함
    return episode


In [51]:
# CartPole 과업

def generateEpisode(env):
    episode = []
    state, info = env.reset()

    image = env.render()
    plt.imshow(image)